[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB03b_training_loop.ipynb)

# NB03b · The training loop: logistic regression and a small MLP
**Artificial Intelligence in Medical Imaging** · Day 2 · Lab 4 (35 min)

| | |
|---|---|
| Runtime | CPU is enough (seconds per run) |
| Data | PneumoniaMNIST, 28 × 28 chest X-rays, 4,708 / 524 / 624 images (train / validation / test), CC BY 4.0 |
| Checkpoints | output shape (batch, 1) · loss goes down · validation loader is not shuffled |
| Tiers | **Core** = TODO cells · **Plus** = learning-rate experiment · **Challenge** = early stopping |

**Learning goals.** (1) Write the four lines every training loop has. (2) Train a one-neuron model (logistic regression) and a small MLP on the pixels. (3) Read a loss curve. (4) Keep the validation set out of training and the test set out of today altogether.

**Rule of the day.** We never look at the test set here. It stays locked until the end of the week.

In [ ]:
%pip install -q "medmnist==3.0.2"
import os, random, time, math
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
for d in ("data", "outputs"): Path(d).mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## 1. Data
Images are scaled to −1…1. Labels: 0 = normal, 1 = pneumonia.

In [ ]:
def load_pneumonia():
    """PneumoniaMNIST (28 px, CC BY 4.0). Falls back to SYNTHETIC data if the download fails."""
    try:
        from medmnist import PneumoniaMNIST
        out = {}
        for s in ("train", "val", "test"):
            d = PneumoniaMNIST(split=s, download=True, root="data")
            out[s] = (d.imgs.reshape(len(d.imgs), 28, 28).astype(np.uint8), d.labels.reshape(-1).astype(np.int64))
        return out, True
    except Exception as e:
        print("!! Could not download PneumoniaMNIST:", repr(e))
        print("!! Using SYNTHETIC data so the code still runs. Numbers below are NOT about pneumonia.")
        rng = np.random.default_rng(0); out = {}
        for s, n in (("train", 2000), ("val", 300), ("test", 300)):
            y = (rng.random(n) < 0.7).astype(np.int64)
            x = rng.normal(100, 25, (n, 28, 28)); x[y == 1, 8:20, 8:20] += 40
            out[s] = (x.clip(0, 255).astype(np.uint8), y)
        return out, False

data, REAL = load_pneumonia()
to_tensor = lambda x: ((torch.from_numpy(x).float() / 255.0 - 0.5) / 0.5).unsqueeze(1)    # (N, 1, 28, 28), range -1..1
X = {s: to_tensor(v[0]) for s, v in data.items()}
Y = {s: torch.from_numpy(v[1]) for s, v in data.items()}
print({s: tuple(X[s].shape) for s in X}, "| real data:", REAL)
maj_val = max(Y["val"].float().mean().item(), 1 - Y["val"].float().mean().item())
print(f"majority-class rate on validation: {maj_val:.3f}  (an 'always the same answer' model scores this)")

In [ ]:
def make_loader(split, bs=128, shuffle=False, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(X[split], Y[split]), batch_size=bs, shuffle=shuffle, generator=g if shuffle else None)

train_loader = make_loader("train", shuffle=True)
val_loader = make_loader("val", shuffle=False)          # never shuffle validation
xb, yb = next(iter(train_loader))
print("batch:", tuple(xb.shape), tuple(yb.shape))
fig, ax = plt.subplots(1, 8, figsize=(14, 2))
for a, i in zip(ax, range(8)): a.imshow(xb[i, 0], cmap="gray"); a.set_title("pneumonia" if yb[i] else "normal", fontsize=8); a.axis("off")
plt.show()

> **Be careful what 28 × 28 means.** These are children's X-rays shrunk to a thumbnail. This is a sandbox for the machinery, not a clinical tool.

## 2. The training loop
Every loop has the same four lines, in this order:

1. clear the old gradients (`opt.zero_grad()`)
2. forward pass and loss
3. backward pass (`loss.backward()`)
4. update the weights (`opt.step()`)

Then evaluate on the validation set **without learning**: `model.eval()` and `torch.no_grad()`.

`evaluate` is given. **TODO 1:** complete `train`.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.float().unsqueeze(1).to(device)
        logits = model(xb)
        loss_sum += criterion(logits, yb).item() * len(xb)
        correct += ((logits > 0).float() == yb).sum().item()          # logit > 0  <=>  probability > 0.5
        n += len(xb)
    return loss_sum / n, correct / n

In [ ]:
def train(model, train_loader, val_loader, epochs=5, lr=3e-3):
    model.to(device)
    criterion = nn.BCEWithLogitsLoss()                    # expects raw scores (logits) and float labels
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"train_loss": [], "val_loss": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        run, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.float().unsqueeze(1).to(device)
            # TODO 1a: clear the old gradients            (hint: opt.zero_grad())

            # TODO 1b: forward pass: logits = model(xb)
            logits = None
            # TODO 1c: loss = criterion(logits, yb)
            loss = None
            # TODO 1d: backward pass                      (hint: loss.backward())

            # TODO 1e: update the weights                 (hint: opt.step())

            run += loss.item() * len(xb); n += len(xb)
        vl, va = evaluate(model, val_loader, criterion)
        hist["train_loss"].append(run / n); hist["val_loss"].append(vl); hist["val_acc"].append(va)
        print(f"epoch {epoch+1:2d} | train loss {run/n:.3f} | val loss {vl:.3f} | val acc {va:.3f}")
    return hist

## 3. Model 1 · logistic regression
One neuron on the 784 pixels: `score = w · pixels + b`, then a sigmoid. Predict first: **how many parameters** does it have? (Write your number, then check below.)

In [ ]:
set_seed(SEED)
logreg = nn.Sequential(nn.Flatten(), nn.Linear(784, 1))
n_params = sum(p.numel() for p in logreg.parameters())
print("parameters:", n_params)
hist_lr = train(logreg, train_loader, val_loader, epochs=5)

### Checkpoint 1

In [ ]:
xb, _ = next(iter(val_loader))
assert logreg(xb.to(device)).shape == (len(xb), 1), "the model must output one score per image: shape (batch, 1)"
assert n_params == 785, "784 weights + 1 bias"
assert hist_lr["train_loss"][-1] < hist_lr["train_loss"][0], "the training loss should go down; check the four lines of the loop"
from torch.utils.data import SequentialSampler
assert isinstance(val_loader.sampler, SequentialSampler), "the validation loader must not be shuffled"
print("Checkpoint 1 passed")

> **Pitfalls to recognise.** `loss` is `nan` → learning rate too high. Loss does not move → learning rate too low, or you forgot `opt.step()`. `RuntimeError: ... Float ... Long` → labels must be float for `BCEWithLogitsLoss`.

## 4. Model 2 · a small MLP
Add one hidden layer of 64 neurons with a ReLU. Predict: how many parameters now? (Compare with 785.)

**TODO 2.** Build the MLP: `Flatten → Linear(784, 64) → ReLU → Linear(64, 1)`.

In [ ]:
set_seed(SEED)
# TODO 2: Flatten -> Linear(784, 64) -> ReLU -> Linear(64, 1)
mlp = None
print("parameters:", sum(p.numel() for p in mlp.parameters()))
hist_mlp = train(mlp, train_loader, val_loader, epochs=5)

In [ ]:
assert sum(p.numel() for p in mlp.parameters()) == 50305, "784*64 + 64 + 64*1 + 1 = 50,305 (the number on the Day-2 slide)"
assert hist_mlp["train_loss"][-1] < hist_mlp["train_loss"][0]
print("Checkpoint 2 passed: 50,305 parameters")

## 5. Curves and comparison

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for name, h in [("logistic regression", hist_lr), ("MLP", hist_mlp)]:
    ax[0].plot(h["train_loss"], label=name); ax[1].plot(h["val_loss"], label=name); ax[2].plot(h["val_acc"], label=name)
for a, t in zip(ax, ["training loss", "validation loss", "validation accuracy"]): a.set_title(t); a.set_xlabel("epoch"); a.legend()
ax[2].axhline(maj_val, color="grey", ls="--"); ax[2].text(0, maj_val + 0.003, "majority-class rate", color="grey")
plt.tight_layout(); plt.show()
n_val = len(Y["val"])
for name, h in [("logistic regression", hist_lr), ("MLP", hist_mlp)]:
    print(f"{name:20s} final val acc {h['val_acc'][-1]:.3f}")
se = (h["val_acc"][-1] * (1 - h["val_acc"][-1]) / n_val) ** 0.5
print(f"sampling noise of one accuracy on {n_val} images is about +/- {100*se:.1f} points")

**Reflection questions** (discuss in your pair, write one sentence each):
1. Your curves: which of the three pictures on the Day-2 slide (under-fit, good fit, over-fit) are they closest to? What would you change?
2. The MLP has 64× more parameters than logistic regression. Is it clearly better? Compare the gap with the noise printed above.
3. A model that answers "pneumonia" for every image scores the majority-class rate. How far above it are you? Why is "accuracy" a weak report here? (Metrics: next lab.)
4. Why is the validation loader not shuffled while the training loader is?

### Plus · learning-rate experiment
Train the logistic regression with `lr` in `[1e-4, 1e-3, 1e-2, 1e-1]` (5 epochs each, fresh model and `set_seed` each time). Report validation accuracy and the final training loss in a table. Which learning rate would you pick and why?

In [ ]:
# [PLUS]
# TODO (Plus): loop over the four learning rates; fresh model and set_seed(SEED) each time;
# collect (lr, final train loss, final val acc) in `rows` and print a table.
rows = []
...

### Challenge · early stopping
Overfitting shows as training loss falling while validation loss rises. **Early stopping** keeps the weights of the best validation epoch and stops after `patience` epochs without improvement. Write `train_es` (reuse `train`'s body) and test it on the MLP with `epochs=30`.

> Selecting the epoch on the *validation* set is legitimate. Selecting it on the test set is not.

In [ ]:
# [CHALLENGE]
# TODO (Challenge): keep a deep copy of model.state_dict() at the best validation loss;
# stop when there is no improvement for `patience` epochs; load the best weights before returning.
def train_es(model, train_loader, val_loader, epochs=30, lr=3e-3, patience=3):
    ...

## Summary
- A training loop = `zero_grad → forward + loss → backward → step`, then evaluate in `eval()` mode under `no_grad`.
- A loss curve is your first diagnostic; the validation set is how you choose, the test set is how you report once.
- More parameters are not automatically better: compare against the noise.

*Next:* NB04 (a convolutional baseline and proper metrics).